# Imputation and encoding 

In [12]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re


import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


In [13]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, "processed_data/20_processed_train_data.csv"))
x_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", x_test.shape)
display(x_test.head(3))


Loaded shape: (75973, 14)


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,Volkswagen,Golf,2016.0,22290.0,semi-auto,28421.0,petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,53000,Toyota,Yaris,2019.0,13790.0,manual,4589.0,petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,6366,Audi,Q2,2019.0,24990.0,semi-auto,3624.0,petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 13)


,carID,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


# Fehlende Werte behandeln

- Numerisch: mit Median auffüllen
- Kategorisch: mit most_frequent auffüllen

In [14]:
# Show missing values
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0]
print("Columns with missing values:\n", missing_values)

Columns with missing values:
 Brand             1521
model             1517
year              1491
transmission      2691
mileage           1463
fuelType          1511
tax               7904
mpg               7926
engineSize        1516
paintQuality%     1524
previousOwners    1550
hasDamage         1548
dtype: int64


In [15]:
# Split df into x and y
x_train = df.drop(columns=["price"])
y_train = df["price"]

In [16]:
# Data types of x_train
print("Data types of x_train:\n", x_train.dtypes)

Data types of x_train:
 carID               int64
Brand              object
model              object
year              float64
transmission       object
mileage           float64
fuelType           object
tax               float64
mpg               float64
engineSize        float64
paintQuality%     float64
previousOwners    float64
hasDamage         float64
dtype: object


In [17]:
from sklearn.impute import SimpleImputer

# Impute numerical columns with median for training set
num_cols = x_train.select_dtypes(include=["int64", "float64"]).columns
num_imputer = SimpleImputer(strategy="median")
x_train[num_cols] = num_imputer.fit_transform(x_train[num_cols])


# Impute categorical columns with most frequent
cat_cols = x_train.select_dtypes(include=["object"]).columns
cat_imputer = SimpleImputer(strategy="most_frequent")
x_train[cat_cols] = cat_imputer.fit_transform(x_train[cat_cols])
print("After imputation, rows with any remaining NaNs:", int(x_train.isna().any(axis=1).sum()))



# Use the same values for the imputation on the test set (no data leakage)
x_test[num_cols] = num_imputer.transform(x_test[num_cols])
x_test[cat_cols] = cat_imputer.transform(x_test[cat_cols])
print("After imputation on test set, rows with any remaining NaNs:", int(x_test.isna().any(axis=1).sum()))


After imputation, rows with any remaining NaNs: 0
After imputation on test set, rows with any remaining NaNs: 0


- Encoding One-Hot for nominal categorical features
- And Ordinal Encoding for ordinal categorical features


In [18]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from packaging import version
import sklearn
import pandas as pd

# --- 1) Spalten sauber bestimmen
all_cat = x_train.select_dtypes(include=["object", "category"]).columns.tolist()

# 👉 Falls du KEINE ordinalen Features hast, lass die Liste leer:
ordinal_cols = []  # z.B. ["education_level", "satisfaction_rating"] falls vorhanden

# Nominale Spalten = alle kategorialen minus der ordinalen
nominal_cols = [c for c in all_cat if c not in ordinal_cols]

# --- 2) One-Hot für nominale Spalten (versionssicher)
ohe_kwargs = dict(drop="first", handle_unknown="ignore")
if version.parse(sklearn.__version__) >= version.parse("1.4"):
    ohe_kwargs["sparse_output"] = False
else:
    ohe_kwargs["sparse"] = False

ohe = OneHotEncoder(**ohe_kwargs)

if len(nominal_cols):
    X_ohe = ohe.fit_transform(x_train[nominal_cols])
    ohe_cols = ohe.get_feature_names_out(nominal_cols)
    x_train = x_train.drop(columns=nominal_cols)
    x_train = pd.concat(
        [x_train, pd.DataFrame(X_ohe, index=x_train.index, columns=ohe_cols)],
        axis=1
    )

# 2.1 One-Hot für test set aber nur transformieren
if len(nominal_cols):
    X_ohe_test = ohe.transform(x_test[nominal_cols])
    ohe_cols = ohe.get_feature_names_out(nominal_cols)
    x_test = x_test.drop(columns=nominal_cols)
    x_test = pd.concat(
        [x_test, pd.DataFrame(X_ohe_test, index=x_test.index, columns=ohe_cols)],
        axis=1
    )
    

# --- 3) Ordinal Encoding (nur wenn vorhanden)
if len(ordinal_cols):
    oe = OrdinalEncoder()
    x_train[ordinal_cols] = oe.fit_transform(x_train[ordinal_cols])
    x_test[ordinal_cols] = oe.transform(x_test[ordinal_cols])


# Skalierung

- Numerische Spalten mit StandardScaler skalieren

In [19]:
from sklearn.preprocessing import StandardScaler

# Numerische Spalten automatisch erkennen
num_cols = x_train.select_dtypes(include=["number"]).columns

# Scaler erstellen
scaler = StandardScaler()

# Auf Trainingsdaten fitten
scaler.fit(x_train[num_cols])

# Danach transformieren
x_train[num_cols] = scaler.transform(x_train[num_cols])
x_test[num_cols]  = scaler.transform(x_test[num_cols])


# Output speicher

In [20]:
# Shape of x_train
print("Shape of x_test after preprocessing:", x_train.shape)
# head of x_train
print(x_train.head())

# Shape of x_test
print("Shape of x_test after preprocessing:", x_test.shape)
# head of x_test
print(x_test.head())



Shape of x_test after preprocessing: (75973, 228)
      carID      year   mileage       tax       mpg  engineSize  paintQuality%  previousOwners  hasDamage  Brand_BMW  Brand_Ford  Brand_Hyundai  \
0  1.437475 -0.501260  0.252019  0.353812 -2.795051    0.600708      -0.076836        1.382747        0.0  -0.328306   -0.548774      -0.214306   
1  0.684586  0.872194 -0.834734  0.353812 -0.458753   -0.279931      -0.701562       -0.686349        0.0  -0.328306   -0.548774      -0.214306   
2 -1.441761  0.872194 -0.878739  0.353812 -0.907022   -0.279931      -0.413227        1.382747        0.0  -0.328306   -0.548774      -0.214306   
3 -0.408772  0.414376 -0.628939  0.353812  0.681132   -1.160570      -0.701562       -2.755444        0.0  -0.328306    1.822245      -0.214306   
4 -1.273236  0.872194 -0.998395  0.353812 -0.785349   -0.279931       1.557065        0.693048        0.0   3.045937   -0.548774      -0.214306   

   Brand_Mercedes-Benz  Brand_Opel  Brand_Toyota  Brand_Volkswagen 

In [21]:
# save the processed dataframe to data/processed_data
PROCESSED_CSV = os.path.join(data_dir, "processed_data/20_processed_train_data.csv")
print("Saving processed file to:", PROCESSED_CSV)
df.to_csv(PROCESSED_CSV, index=False)
print("✅ Saved. Rows with any remaining NaNs:", int(df.isna().any(axis=1).sum()))



Saving processed file to: ../data/processed_data/20_processed_train_data.csv
✅ Saved. Rows with any remaining NaNs: 22402


In [22]:
# Saved the encoded data (x_train and x_test) and y_train to data/encoded_data
ENCODED_DIR = os.path.join(data_dir, "encoded_data")
os.makedirs(ENCODED_DIR, exist_ok=True) 

X_TRAIN_CSV = os.path.join(ENCODED_DIR, "x_train_encoded.csv")
Y_TRAIN_CSV = os.path.join(ENCODED_DIR, "y_train.csv")
X_TEST_CSV  = os.path.join(ENCODED_DIR, "x_test_encoded.csv")

x_train.to_csv(X_TRAIN_CSV, index=False)
y_train.to_csv(Y_TRAIN_CSV, index=False)
x_test.to_csv(X_TEST_CSV, index=False)
print("✅ Saved encoded x_train to:", X_TRAIN_CSV)
print("✅ Saved y_train to:", Y_TRAIN_CSV)
print("✅ Saved encoded x_test to:", X_TEST_CSV)

✅ Saved encoded x_train to: ../data/encoded_data/x_train_encoded.csv
✅ Saved y_train to: ../data/encoded_data/y_train.csv
✅ Saved encoded x_test to: ../data/encoded_data/x_test_encoded.csv
